# Stage 20 Final Held-Out Test Evaluation

This notebook is the guarded final test-set evaluation workflow. Phase 1 prepares and freezes the candidate registry from validation artifacts only. Phase 2 should be run once, after Stage 19 has finished and the registry is frozen. The held-out test split is not evaluated unless `RUN_FINAL_TEST_EVALUATION` is explicitly set to `True`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.final_evaluation import (
    DEFAULT_STAGE20_OUTPUT_DIR,
    FINAL_TEST_GUARD_MESSAGE,
    assert_final_registry_ready,
    build_final_candidate_registry,
    freeze_candidate_registry,
    load_prediction_tables,
    require_final_test_confirmation,
    run_final_test_evaluation,
)

stage20_output_dir = repo_root / DEFAULT_STAGE20_OUTPUT_DIR
prediction_dir = stage20_output_dir / "predictions"
stage20_output_dir.mkdir(parents=True, exist_ok=True)
prediction_dir.mkdir(parents=True, exist_ok=True)

{
    "repo_root": str(repo_root),
    "stage20_output_dir": str(stage20_output_dir),
    "prediction_dir": str(prediction_dir),
}


## Candidate Registry

The registry is built from validation artifacts only. Stage 19 will remain pending until its summary exists, and every candidate must be `ready` before the final test run is allowed.

In [ ]:
registry = build_final_candidate_registry(results_dir=repo_root / "results")
registry.to_csv(stage20_output_dir / "candidate_registry_draft.csv", index=False)
display(registry)

pending = registry[registry["status"] != "ready"]
if pending.empty:
    print("All candidates are ready to freeze.")
else:
    print("Pending candidates:")
    display(pending[["candidate_id", "status", "notes"]])


## Freeze Registry

Set `FREEZE_FINAL_CANDIDATE_REGISTRY = True` only after Stage 19 has completed and the displayed candidate list is the final pre-test list.

In [ ]:
FREEZE_FINAL_CANDIDATE_REGISTRY = False

if FREEZE_FINAL_CANDIDATE_REGISTRY:
    paths = freeze_candidate_registry(registry, output_dir=stage20_output_dir)
    print("Frozen registry:", paths["registry"])
    print("Manifest:", paths["manifest"])
else:
    print("Final candidate registry not frozen in this run.")


## Final Test Guard

The final evaluation cell expects validation and test prediction CSVs to be materialized under `results/stage20_final_test_evaluation/predictions/`. Leave `RUN_FINAL_TEST_EVALUATION = False` until the registry is frozen and the prediction artifacts have been generated for the locked candidates.

In [ ]:
RUN_FINAL_TEST_EVALUATION = False

if RUN_FINAL_TEST_EVALUATION:
    require_final_test_confirmation(RUN_FINAL_TEST_EVALUATION)
    frozen_registry_path = stage20_output_dir / "final_candidate_registry.csv"
    frozen_registry = pd.read_csv(frozen_registry_path)
    assert_final_registry_ready(frozen_registry)

    validation_prediction_paths = sorted(prediction_dir.glob("validation_predictions_*.csv"))
    test_prediction_paths = sorted(prediction_dir.glob("test_predictions_*.csv"))
    validation_predictions = load_prediction_tables(validation_prediction_paths)
    test_predictions = load_prediction_tables(test_prediction_paths)

    outputs = run_final_test_evaluation(
        registry=frozen_registry,
        validation_predictions=validation_predictions,
        test_predictions=test_predictions,
        output_dir=stage20_output_dir,
        run_final_test=True,
        make_plots=True,
    )
    display(outputs["validation_test_metric_comparison"])
    display(outputs["duration_error_summary"])
else:
    print(FINAL_TEST_GUARD_MESSAGE)
